In [11]:
import pandas as pd
df = pd.read_csv('dim_clientes_202606161344.csv')
df.head()
import numpy as np

# 1. Carregando a tabela Fato (substitua pelo nome exato do seu arquivo CSV)
df_fato = pd.read_csv('fato_churn_202606161352.csv')

# 2. Convertendo a coluna de data para o formato de data do Python
df_fato['DataUltimaTransacao'] = pd.to_datetime(df_fato['DataUltimaTransacao'])

# 3. Criando a regra de negócio do Churn:
# A extração foi em 30/10/2019. Quem não comprou nos últimos 30 dias (antes de 30/09/2019) é Churn (1).
data_de_corte = pd.to_datetime('2019-09-30')
df_fato['Churn'] = np.where(df_fato['DataUltimaTransacao'] < data_de_corte, 1, 0)

# 4. Juntando as tabelas usando o 'Clientid' como chave (como se fosse um PROCV/VLOOKUP)
# Assumindo que a sua tabela dim_clientes ainda se chama 'df'
df_master = pd.merge(df, df_fato[['ClientId', 'Churn']], on='ClientId', how='left')

# Visualizando se a coluna 'Churn' apareceu no final
df_master.head()

,ClientId,DataExtracao,Score_Credito,Estado,Gênero,Idade,Tempo_Cliente,Limite_Credito_Mercado,Qte_Categorias,Usa_Cartao_Credito,Programa_Fidelidade,Sum_Pedidos_Acumulados,Churn
0,345568,2019-06-30,619,São Paulo,Feminino,42,2,0.0000,1,1,1,422.287000,1
1,345569,2019-06-30,608,Rio de Janeiro,Feminino,41,1,838.0786,1,0,1,468.927417,0
2,345570,2019-06-30,502,São Paulo,Feminino,42,8,1596.6080,3,1,0,474.714875,1
3,345571,2019-06-30,699,São Paulo,Feminino,39,1,0.0000,2,0,0,390.944292,0
4,345572,2019-06-30,850,Rio de Janeiro,Feminino,43,2,1255.1082,1,1,1,329.517083,0


In [12]:
# Calculando a correlação apenas para as colunas numéricas e focando na coluna 'Churn'
correlacao = df_master.corr(numeric_only=True)[['Churn']].sort_values(by='Churn', ascending=False)

# Exibindo o resultado
correlacao

,Churn
Churn,1.000000
Idade,0.283237
Limite_Credito_Mercado,0.116746
Sum_Pedidos_Acumulados,0.012198
Usa_Cartao_Credito,-0.006112
Tempo_Cliente,-0.013174
ClientId,-0.016589
Score_Credito,-0.028936
Qte_Categorias,-0.049865
Programa_Fidelidade,-0.154643


In [13]:
!pip install scorecardpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for scorecardpy: filename=scorecardpy-0.1.9.7-py3-none-any.whl size=60629 sha256=050eb403ae452d8a20c5067fb27fcfcb54391ead688faf3de5465dd117ed32ba
  Stored in directory: /root/.cache/pip/wheels/9f/d8/4e/61a6f4e78fe6700f66b699ab38377f0aa5b33e3ef55751ba38
Successfully built scorecardpy


In [14]:
import scorecardpy as sc

# Calculando o IV para todas as variáveis em relação ao 'Churn'
# Ignoramos a coluna 'Clientid' pois ela é apenas um código de identificação e não tem valor analítico
info_value = sc.iv(df_master.drop(columns=['ClientId']), y="Churn")

# Exibindo os resultados do maior para o menor poder preditivo
info_value

,variable,info_value
4,Qte_Categorias,0.931941
9,Idade,0.910873
5,Limite_Credito_Mercado,0.463426
0,Score_Credito,0.268104
3,Estado,0.166786
10,Programa_Fidelidade,0.151655
8,Gênero,0.072979
1,Tempo_Cliente,0.009086
2,Sum_Pedidos_Acumulados,0.007218
6,Usa_Cartao_Credito,0.000231


In [17]:
from sklearn.tree import DecisionTreeClassifier
import numpy as np

# 1. Separando as variáveis que o modelo vai usar para aprender (X) e o que ele precisa prever (y)
colunas_importantes = ['Idade', 'Limite_Credito_Mercado', 'Score_Credito', 'Qte_Categorias', 'Programa_Fidelidade']
X = df_master[colunas_importantes]
y = df_master['Churn']

# 2. Criando e treinando o Algoritmo (Árvore de Decisão)
modelo = DecisionTreeClassifier(max_depth=4, random_state=42)
modelo.fit(X, y)

# 3. Calculando a probabilidade de cada cliente dar Churn (de 0 a 1) e transformando em porcentagem
df_master['Probabilidade_Churn_%'] = modelo.predict_proba(X)[:, 1] * 100

# 4. Criando as regras de Segmentação para o time de Negócios/CRM
condicoes = [
    (df_master['Probabilidade_Churn_%'] >= 70),
    (df_master['Probabilidade_Churn_%'] >= 30) & (df_master['Probabilidade_Churn_%'] < 70),
    (df_master['Probabilidade_Churn_%'] < 30)
]
segmentos = ['Alto Risco', 'Médio Risco', 'Baixo Risco']

# CORREÇÃO APLICADA AQUI: Adicionado o default='Sem Classificação'
df_master['Segmento_Risco'] = np.select(condicoes, segmentos, default='Sem Classificação')

# 5. Contando quantos clientes temos em cada segmento
contagem_segmentos = df_master['Segmento_Risco'].value_counts()
print("Distribuição de Clientes por Risco:")
print(contagem_segmentos)

Distribuição de Clientes por Risco:
Segmento_Risco
Baixo Risco    7997
Médio Risco    1424
Alto Risco      579
Name: count, dtype: int64


In [18]:
# Exportando para Excel (ideal para abrir no seu computador e conferir)
df_master.to_excel('resultado_churn_tocomfome.xlsx', index=False)

# Ou exportando para CSV
# df_master.to_csv('resultado_churn_tocomfome.csv', index=False)